# 支付方式与分期

看不同支付方式的订单量和客单价，以及分期对客单价的影响。

In [1]:
import pandas as pd
from db import query

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
query("""
SELECT main_payment_type,
       COUNT(*) AS orders,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS pct,
       ROUND(AVG(payment_value), 2) AS avg_value,
       ROUND(SUM(payment_value), 2) AS total_value
FROM dwd_order_payment
GROUP BY main_payment_type
ORDER BY orders DESC
""")

,main_payment_type,orders,pct,avg_value,total_value
0,credit_card,74975,75.40,167.29,12542527.15
1,boleto,19784,19.90,145.03,2869361.27
2,voucher,3151,3.17,120.29,379043.91
3,debit_card,1527,1.54,142.72,217939.79
4,not_defined,3,0.00,0.00,0.00


In [3]:
query("""
SELECT CASE
         WHEN max_installments = 1 THEN '不分期'
         WHEN max_installments <= 3 THEN '2-3期'
         WHEN max_installments <= 6 THEN '4-6期'
         ELSE '6期以上'
       END AS installment_bucket,
       COUNT(*) AS orders,
       ROUND(AVG(payment_value), 2) AS avg_value
FROM dwd_order_payment
GROUP BY installment_bucket
ORDER BY orders DESC
""")

,installment_bucket,orders,avg_value
0,不分期,48268,121.04
1,2-3期,22794,136.09
2,4-6期,16205,182.68
3,6期以上,12173,337.16


分期越长客单价越高，符合大额消费分摊的心理，可以把分期作为高客单类目的转化工具。